# PBL6 - Notebook 1/2: TIỀN XỬ LÝ DỮ LIỆU DFDC (đọc từ Google Drive)

**Việc notebook này làm:** đọc dataset DFDC có sẵn trên Drive (các file `.zip` hoặc thư mục đã giải nén) → chỉ giải nén **từng video cần dùng** → tách frame + audio bằng ffmpeg → chia train/valid/test → nén thành **một file `.tar`** lưu lại lên Drive.

**Notebook 2 (`pbl6_2_train.ipynb`) chỉ việc giải nén file `.tar` này rồi train.**

**Trước khi chạy:**
1. Runtime → Change runtime type → **CPU** (không cần GPU).
2. Sửa `DATASET_DIR` ở cell số 2 cho đúng thư mục dataset trên Drive của bạn.
3. Chạy lần lượt từ trên xuống.

**Kết quả:** `MyDrive/pbl6_mouth/dfdc_processed.tar` (bản crop **miệng**; thư mục riêng nên không ghi đè dữ liệu crop mặt cũ)

In [ ]:
import os
print('CPU (nproc):', os.popen('nproc').read().strip())
print('CPU thuc dung duoc:', len(os.sched_getaffinity(0)))
!lscpu | grep -E "Model name|^CPU\(s\)"
!free -h | head -2

In [ ]:
# Mount Google Drive (noi luu file ket qua cuoi cung)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/pbl6', exist_ok=True)

## 1. Chỉ đường dẫn dataset trên Drive và số lượng cần lấy

In [ ]:
import os, glob

# ===== SUA CHO DUNG =====
# Thu muc tren Drive chua cac video (.mp4) + 1 file metadata.json (cung la thu muc). Cung ho tro thu muc chua cac file .zip.
# Neu dataset nam trong Drive cua tai khoan khac (qua loi tat/shortcut) thi van dung duoc duong dan cua shortcut.
DATASET_DIR = '/content/drive/MyDrive/dfdc_real_fake15'

# Lay HET video: dat None. Muon gioi han (VD ty le REAL:FAKE = 40:60 thi MAX_FAKE = 1.5 * MAX_REAL) thi dat so.
MAX_REAL = 13000
MAX_FAKE = 13000
MAX_CHUNKS = 5        # so doan 1 giay toi da lay tu moi video (None = lay het cac doan)
WORKERS = 8           # so video xu ly song song (che do thu muc). Colab ~2 CPU: 3-4 la hop ly
# =========================

assert os.path.isdir(DATASET_DIR), f'Khong thay {DATASET_DIR}. Kiem tra lai duong dan / mount Drive.'
zips = sorted(glob.glob(os.path.join(DATASET_DIR, '**', '*.zip'), recursive=True))
metas = sorted(glob.glob(os.path.join(DATASET_DIR, '**', '*.json'), recursive=True))
print('Thu muc:', DATASET_DIR)
print(f'Tim thay {len(zips)} file zip, {len(metas)} file .json')
for m in metas[:5]:
    print('  json:', m)
vids = glob.glob(os.path.join(DATASET_DIR, '**', '*.mp4'), recursive=True)
print(f'So video .mp4 trong thu muc: {len(vids)}')
for z in zips[:5]:
    print('  ', os.path.basename(z), f'{os.path.getsize(z)/1024**3:.2f} GB')
if len(zips) > 5:
    print('   ...')

## 2. Cài ffmpeg (tách frame/audio) và MediaPipe (landmark môi)

In [ ]:
!pip uninstall -y -q imageio-ffmpeg
!pip install -q "imageio-ffmpeg==0.4.5"
# MediaPipe: chi can khi CROP_MODE = 'mouth_lm'. Neu Colab bao can khoi dong lai runtime thi khoi dong lai roi chay lai tu dau.
!pip install -q mediapipe

In [ ]:
import imageio_ffmpeg, os

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
print('ffmpeg binary tại:', ffmpeg_path)

os.makedirs('/usr/local/bin', exist_ok=True)
symlink_path = '/usr/local/bin/ffmpeg'
if os.path.exists(symlink_path) or os.path.islink(symlink_path):
    os.remove(symlink_path)
os.symlink(ffmpeg_path, symlink_path)

import subprocess
result = subprocess.run(['ffmpeg', '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print('ffmpeg return code:', result.returncode)
print(result.stdout.decode()[:200])

## 3. Import và cấu hình

In [ ]:
import os
import sys
import glob
import json
import csv
import time
import random
import shutil
import subprocess
from datetime import datetime

import pandas as pd
from tqdm import tqdm

print('Imports OK')

In [ ]:
class Config:
    # Thu muc lam viec TREN O CUC BO cua Colab (nhanh). Mat khi Colab ngat -> ket qua cuoi se duoc nen luu len Drive.
    WORK_DIR = '/content/work'
    DRIVE_DIR = '/content/drive/MyDrive/pbl6_mouth'   # thu muc RIENG cho ban crop mieng (khong de len du lieu crop mat cu)
    TRAIN_DIR = os.path.join(WORK_DIR, 'train')

    # Ty le chia THEO VIDEO (phan con lai la test)
    TRAIN_RATIO = 0.85
    VALID_RATIO = 0.1
    SEED = 0

cfg = Config()
os.makedirs(cfg.WORK_DIR, exist_ok=True)
print('Configuration loaded.')
print(f'  Dataset    : {DATASET_DIR}')
print(f'  Working Dir: {cfg.WORK_DIR}')
print(f'  Drive Dir  : {cfg.DRIVE_DIR}')
print(f'  Split      : train {cfg.TRAIN_RATIO} / valid {cfg.VALID_RATIO} / test {round(1-cfg.TRAIN_RATIO-cfg.VALID_RATIO,3)}')

## 4. Tiền xử lý video → crop mặt + lọc chuyển cảnh → frame + audio

Mỗi video được đổi sang 30 fps, tách thành các đoạn 1 giây (30 frame + 1 giây audio WAV 48 kHz mono). Với mỗi đoạn:

1. **Phát hiện chuyển cảnh:** so histogram màu giữa các frame liên tiếp; đoạn nào có cắt cảnh thì bỏ.
2. **Phát hiện mặt:** dò mặt ở 5 frame mẫu (YuNet, tự lùi về Haar nếu không tải được model). Đoạn nào không đủ mặt, hoặc mặt di chuyển quá nhiều / đổi người thì bỏ.
3. **Crop vùng miệng** (`CROP_MODE = 'mouth_lm'`): YuNet tìm mặt, rồi **MediaPipe Face Landmarker** lấy 40 điểm landmark môi trong vùng mặt đó. Ô vuông có tâm là tâm môi, cạnh `LIP_SCALE` × bề rộng môi (từ dưới mũi đến cằm). Nếu một frame mẫu không lấy được landmark hợp lý thì frame đó dùng ô theo 2 mép miệng của YuNet. Hợp nhất các ô của 5 frame mẫu thành **một ô vuông** dùng cho cả 30 frame (giữ nhất quán theo thời gian), rồi resize 224x224. `CROP_MODE = 'mouth'` là bản không cần MediaPipe (ô theo cạnh mặt, `MOUTH_SCALE`); `'face'` là crop cả mặt. Lần chạy đầu cell này sẽ tải file model `face_landmarker.task` (vài MB) và **báo lỗi nếu thiếu mediapipe**, không âm thầm đổi cách crop.

Mỗi video quét tối đa `SCAN_SECONDS` giây đầu và giữ lại tối đa `MAX_CHUNKS` đoạn hợp lệ. Cell cuối của mục này in thống kê số đoạn bị loại và lý do.

**Lưu ý:** dữ liệu tạo ra khác hẳn bản crop mặt cũ (nay là ảnh **chỉ có vùng miệng**) nên phải **train lại notebook 2**, và `predict_video` ở notebook 2 phải dùng đúng cùng cách xử lý.

In [ ]:
# ======================= LOI: PHAT HIEN MAT + CHUYEN CANH =======================
import os, shutil, subprocess, wave, threading, urllib.request
from collections import Counter   # khong dung "import collections": notebook 2 da alias ten do sang collections.abc
import numpy as np
import cv2

# ---- tham so (co the chinh) ----
SCAN_SECONDS     = 8      # quet toi da bay nhieu giay dau cua video de tim doan hop le
FRAME_SHORT_SIDE = 540    # canh ngan cua frame trung gian (de crop mat; lon hon = net hon nhung cham hon)
CUT_THRESH       = 0.45   # khoang cach Bhattacharyya giua 2 frame lien tiep > nguong => co chuyen canh (tang nguong = it loai hon)
FACE_SAMPLE_IDX  = (0, 7, 15, 22, 29)   # cac frame (trong 30 frame cua doan) dung de do mat
MIN_FACE_FRAC    = 0.6    # can tim thay mat o it nhat 60% so frame mau
CROP_MODE        = 'mouth_lm'   # 'mouth_lm' = YuNet tim mat + MediaPipe lay landmark MOI -> o crop bam sat mieng that (can mediapipe)
                                # 'mouth'    = o vuong theo kich thuoc mat + 2 mep mieng cua YuNet (khong can mediapipe)
                                # 'face'     = crop ca mat (cach cu)
MIN_FACE_PX      = 64 if CROP_MODE in ('mouth', 'mouth_lm') else 48   # canh mat toi thieu (frame trung gian); o crop mieng nho hon nen can mat lon hon cho du net
MAX_UNION_RATIO  = 2.0    # vung chua tat ca mat mau / kich thuoc mat trung binh; lon hon => mat di chuyen qua nhieu hoac doi nguoi
FACE_MARGIN      = 1.4    # he so mo rong box mat (1.0 = sat mat; 1.4 = lay them tran/cam/tai) - chi dung khi CROP_MODE = 'face'
MOUTH_SCALE      = 0.7    # canh o vuong quanh mieng = MOUTH_SCALE * canh mat (0.7: tu duoi mui den cam; 0.6 = chat hon, 0.8 = rong hon)
MOUTH_Y_SHIFT    = 0.05   # day tam o crop xuong duoi (ti le theo canh o) de co them ham/cam
MOUTH_MARGIN     = 1.1    # he so mo rong sau khi hop nhat cac o mieng trong 1 doan (mat dich chuyen nhe van khong cham mep)
LIP_SCALE         = 1.6   # ('mouth_lm') canh o = LIP_SCALE * be rong moi do tu 40 diem landmark (1.6: tu duoi mui den cam; 1.4 = chat hon, 1.8 = rong hon)
LIP_MIN_FACE_FRAC = 0.5   # ('mouth_lm') o khong nho hon 50% canh mat (tranh qua chat khi chum moi "u")
LANDMARKER_URL    = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
LANDMARKER_PATH   = '/tmp/face_landmarker.task'
OUT_SIZE         = 224
JPEG_QUALITY     = 95

YUNET_URL  = 'https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx'
YUNET_PATH = '/tmp/face_detection_yunet_2023mar.onnx'

# ---- thong ke (de biet bao nhieu doan bi loai va vi sao) ----
STATS = Counter()
_stats_lock = threading.Lock()
def _stat(key):
    with _stats_lock:
        STATS[key] += 1

# ---- bo do mat: YuNet (nhanh tren CPU); neu khong tai duoc thi dung Haar cua OpenCV ----
_model_lock = threading.Lock()
_yunet_ok = None
def _yunet_available():
    global _yunet_ok
    with _model_lock:
        if _yunet_ok is not None:
            return _yunet_ok
        try:
            if not (os.path.exists(YUNET_PATH) and os.path.getsize(YUNET_PATH) > 100_000):
                urllib.request.urlretrieve(YUNET_URL, YUNET_PATH)
            ok = os.path.getsize(YUNET_PATH) > 100_000 and hasattr(cv2, 'FaceDetectorYN')
        except Exception as e:
            print('Khong tai duoc YuNet, chuyen sang Haar (kem hon):', e)
            ok = False
        _yunet_ok = ok
        print('Bo do mat:', 'YuNet' if ok else 'Haar cascade')
        return ok

class _FaceDetector:
    def __init__(self):
        self.yunet = None
        self.haar = None
        if _yunet_available():
            self.yunet = cv2.FaceDetectorYN.create(YUNET_PATH, '', (320, 320), 0.7, 0.3, 5000)
        else:
            self.haar = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    def detect(self, bgr):
        """Tra ve list (x, y, w, h, tam_mieng_x, tam_mieng_y) theo toa do cua anh dau vao."""
        h, w = bgr.shape[:2]
        if self.yunet is not None:
            self.yunet.setInputSize((w, h))
            _, faces = self.yunet.detect(bgr)
            if faces is None:
                return []
            # f = [x, y, w, h, mat phai(2), mat trai(2), mui(2), mep mieng phai(2), mep mieng trai(2), score]
            return [(float(f[0]), float(f[1]), float(f[2]), float(f[3]),
                     (float(f[10]) + float(f[12])) / 2, (float(f[11]) + float(f[13])) / 2) for f in faces]
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        found = self.haar.detectMultiScale(gray, 1.1, 5, minSize=(24, 24))
        out = []
        for (x, y, w, h) in found:    # Haar khong co diem moc -> uoc luong tam mieng ~78% chieu cao mat
            out.append((float(x), float(y), float(w), float(h), float(x) + w / 2.0, float(y) + 0.78 * h))
        return out

_tls = threading.local()
def _get_detector():
    if not hasattr(_tls, 'det'):          # moi luong 1 bo do rieng (OpenCV detector khong an toan da luong)
        _tls.det = _FaceDetector()
    return _tls.det

# ---- doc frame / phat hien chuyen canh ----
def _fpath(frames_dir, i):
    return os.path.join(frames_dir, '%06d.jpg' % i)

def _read_small(path):
    return cv2.imread(path, cv2.IMREAD_REDUCED_COLOR_2)   # giai ma nua kich thuoc -> nhanh gap nhieu lan

def _hist(small):
    t = cv2.resize(small, (64, 36), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(t, cv2.COLOR_BGR2HSV)
    h = cv2.calcHist([hsv], [0, 1], None, [16, 8], [0, 180, 0, 256])
    cv2.normalize(h, h, 1, 0, cv2.NORM_L1)
    return h

def has_scene_cut(smalls):
    prev, worst = None, 0.0
    for s in smalls:
        h = _hist(s)
        if prev is not None:
            worst = max(worst, cv2.compareHist(prev, h, cv2.HISTCMP_BHATTACHARYYA))
        prev = h
    return worst > CUT_THRESH, worst

# ---- chon mat + tinh box crop cho ca doan ----
def _pick_face(faces, prev_center):
    if not faces:
        return None
    if prev_center is None:
        return max(faces, key=lambda f: f[2] * f[3])
    return min(faces, key=lambda f: (f[0] + f[2] / 2 - prev_center[0]) ** 2 + (f[1] + f[3] / 2 - prev_center[1]) ** 2)

# ---- MediaPipe Face Landmarker (API Tasks): lay 40 diem landmark MOI trong vung mat YuNet tim duoc. Chi dung khi CROP_MODE = 'mouth_lm' ----
_lm_lock = threading.Lock()
_lm_ready = None
LM_STATS = Counter()      # 'lm_ok' = o crop tu landmark moi; 'lm_fallback' = khong lay duoc -> dung o theo mep mieng YuNet (khong tinh vao STATS)

def _lip_indices():
    from mediapipe.tasks.python import vision
    conns = vision.FaceLandmarksConnections.FACE_LANDMARKS_LIPS
    return sorted({i for c in conns for i in (c.start, c.end)})

def _get_landmarker():
    if not hasattr(_tls, 'lm'):          # moi luong 1 bo rieng (khong dung chung giua cac luong)
        from mediapipe.tasks.python import vision, BaseOptions
        opts = vision.FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=LANDMARKER_PATH),
            running_mode=vision.RunningMode.IMAGE, num_faces=1,
            min_face_detection_confidence=0.3, min_face_presence_confidence=0.3)
        _tls.lm = vision.FaceLandmarker.create_from_options(opts)
        _tls.lip_idx = np.array(_lip_indices())
    return _tls.lm, _tls.lip_idx

def _landmarker_available():
    global _lm_ready
    with _lm_lock:
        if _lm_ready is not None:
            return _lm_ready
        try:
            import mediapipe                      # noqa: F401  (bao loi som neu chua cai)
            if not (os.path.exists(LANDMARKER_PATH) and os.path.getsize(LANDMARKER_PATH) > 1_000_000):
                urllib.request.urlretrieve(LANDMARKER_URL, LANDMARKER_PATH)
            ok = os.path.getsize(LANDMARKER_PATH) > 1_000_000
            if ok:
                _get_landmarker()                 # thu tao that de chac chan file model dung
        except Exception as e:
            print('Khong dung duoc MediaPipe Face Landmarker:', repr(e))
            ok = False
        _lm_ready = ok
        print('Bo landmark moi:', 'MediaPipe Face Landmarker' if ok else 'KHONG CO')
        return ok

def _lip_pts(rgb):
    """Tra ve mang (40, 2) toa do CHUAN HOA (0..1, theo anh dua vao) cua cac diem moi, hoac None."""
    import mediapipe as mp
    lm, idx = _get_landmarker()
    res = lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)))
    if not res.face_landmarks:
        return None
    p = res.face_landmarks[0]
    return np.array([(p[i].x, p[i].y) for i in idx], dtype=np.float32)

def _lip_box_from_pts(pts, crop_rect, f):
    """pts: diem moi chuan hoa trong o crop_rect=(x1, y1, cw, ch); f = (x, y, w, h, mx, my) cua mat. Tra ve o vuong (x, y, w, h) quanh MOI hoac None."""
    if pts is None:
        return None
    x1, y1, cw, ch = crop_rect
    px = pts[:, 0] * cw + x1
    py = pts[:, 1] * ch + y1
    lx1, lx2, ly1, ly2 = float(px.min()), float(px.max()), float(py.min()), float(py.max())
    lip_w = lx2 - lx1
    face_side = 0.5 * (f[2] + f[3])
    cx, cy = (lx1 + lx2) / 2, (ly1 + ly2) / 2
    # loai ket qua bat thuong: moi qua be/lon so voi mat, hoac lech xa vi tri mieng ma bo do mat uoc luong (co the bat nham mat khac)
    if not (0.2 * face_side <= lip_w <= 0.8 * face_side):
        return None
    if ((cx - f[4]) ** 2 + (cy - f[5]) ** 2) ** 0.5 > 0.25 * face_side:
        return None
    side = max(LIP_SCALE * lip_w, LIP_MIN_FACE_FRAC * face_side)
    cy = cy + MOUTH_Y_SHIFT * side
    return (cx - side / 2, cy - side / 2, side, side)

def _lip_box(full_bgr, f):
    H, W = full_bgr.shape[:2]
    x, y, w, h = f[:4]
    m = 0.25 * max(w, h)                       # mo rong box mat de landmarker thay du dau/cam
    x1, y1 = int(max(0, x - m)), int(max(0, y - m))
    x2, y2 = int(min(W, x + w + m)), int(min(H, y + h + m))
    if x2 - x1 < 16 or y2 - y1 < 16:
        return None
    pts = _lip_pts(cv2.cvtColor(full_bgr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
    return _lip_box_from_pts(pts, (x1, y1, x2 - x1, y2 - y1), f)

def _sample_box(frames_dir, idx, f):
    """O crop cho 1 frame mau (toa do frame trung gian) theo CROP_MODE."""
    if CROP_MODE == 'mouth_lm':
        full = cv2.imread(_fpath(frames_dir, idx))
        b = _lip_box(full, f) if full is not None else None
        with _stats_lock:
            LM_STATS['lm_ok' if b is not None else 'lm_fallback'] += 1
        return b if b is not None else _mouth_box(f)
    if CROP_MODE == 'mouth':
        return _mouth_box(f)
    return f[:4]

if CROP_MODE == 'mouth_lm' and not _landmarker_available():
    raise RuntimeError("CROP_MODE = 'mouth_lm' nhung khong dung duoc MediaPipe Face Landmarker (xem thong bao o tren). "
                       "Cach xu ly: !pip install -q mediapipe (xong khoi dong lai runtime neu Colab yeu cau), kiem tra mang tai duoc "
                       "file model, hoac dat CROP_MODE = 'mouth' (khong can mediapipe). Khong duoc tu doi cach crop giua luc train va suy luan.")

def _mouth_box(f):
    """Tu (x, y, w, h, mouth_cx, mouth_cy) cua 1 khuon mat -> o vuong (x, y, w, h) quanh MIENG.
    Canh o tinh theo kich thuoc mat (on dinh hon do theo 2 mep mieng vi diem moc co the nhieu)."""
    x, y, w, h, mx, my = f
    side = MOUTH_SCALE * 0.5 * (w + h)
    cy = my + MOUTH_Y_SHIFT * side
    return (mx - side / 2, cy - side / 2, side, side)

def analyze_chunk(frames_dir, start):
    """Phan tich doan 30 frame bat dau tu frame so `start` (danh so tu 1).
    Tra ve (box, ly_do). box = (cx, cy, canh) tren frame trung gian, hoac None neu doan bi loai."""
    smalls = [_read_small(_fpath(frames_dir, i)) for i in range(start, start + 30)]
    if any(s is None for s in smalls):
        return None, 'doc_frame_loi'
    cut, _ = has_scene_cut(smalls)
    if cut:
        return None, 'chuyen_canh'
    det = _get_detector()
    boxes, prev = [], None
    for k in FACE_SAMPLE_IDX:
        faces = [tuple(v * 2 for v in f) for f in det.detect(smalls[k])]   # x2: ve toa do frame trung gian
        faces = [f for f in faces if min(f[2], f[3]) >= MIN_FACE_PX]
        f = _pick_face(faces, prev)
        if f is not None:
            boxes.append(_sample_box(frames_dir, start + k, f))
            prev = (f[0] + f[2] / 2, f[1] + f[3] / 2)    # van theo doi theo tam MAT (de khong nhay sang nguoi khac)
    if len(boxes) < MIN_FACE_FRAC * len(FACE_SAMPLE_IDX):
        return None, 'khong_co_mat'
    x1 = min(b[0] for b in boxes); y1 = min(b[1] for b in boxes)
    x2 = max(b[0] + b[2] for b in boxes); y2 = max(b[1] + b[3] for b in boxes)
    mean_side = float(np.mean([max(b[2], b[3]) for b in boxes]))
    extent = max(x2 - x1, y2 - y1)
    if extent > MAX_UNION_RATIO * mean_side:
        return None, 'mat_di_chuyen_nhieu'
    margin = MOUTH_MARGIN if CROP_MODE in ('mouth', 'mouth_lm') else FACE_MARGIN
    return ((x1 + x2) / 2, (y1 + y2) / 2, extent * margin), 'ok'

def crop_chunk(frames_dir, start, box, out_size=OUT_SIZE):
    """Crop cung 1 o vuong cho ca 30 frame (giu nhat quan theo thoi gian). Tra ve list anh BGR out_size x out_size."""
    cx, cy, side = box
    first = cv2.imread(_fpath(frames_dir, start))
    H, W = first.shape[:2]
    s = int(round(min(side, H, W)))
    x1 = int(round(cx - s / 2)); y1 = int(round(cy - s / 2))
    x1 = max(0, min(x1, W - s)); y1 = max(0, min(y1, H - s))
    out = []
    for i in range(start, start + 30):
        img = first if i == start else cv2.imread(_fpath(frames_dir, i))
        out.append(cv2.resize(img[y1:y1 + s, x1:x1 + s], (out_size, out_size), interpolation=cv2.INTER_AREA))
    return out

# ---- ffmpeg: tach frame (giu ti le khung hinh) + audio ----
def extract_frames_audio(video_path, work_dir, seconds=SCAN_SECONDS):
    os.makedirs(work_dir, exist_ok=True)
    audio_path = os.path.join(work_dir, 'audio.wav')
    cap = cv2.VideoCapture(video_path)
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    if W > 0 and H > 0 and min(W, H) > FRAME_SHORT_SIDE:
        f = FRAME_SHORT_SIDE / min(W, H)
        W, H = int(round(W * f / 2) * 2), int(round(H * f / 2) * 2)
    vf = f'scale={W}:{H}' if W > 0 and H > 0 else f'scale=-2:{FRAME_SHORT_SIDE}'
    t = ['-t', str(seconds)]
    cmd = (['ffmpeg', '-y', '-v', 'error', '-i', video_path]
           + t + ['-r', '30', '-qscale:v', '2', '-vf', vf, '-threads', '1', '-f', 'image2', os.path.join(work_dir, '%06d.jpg')]
           + t + ['-vn', '-ac', '1', '-acodec', 'pcm_s16le', '-ar', '48000', audio_path])
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=300)
    n_frames = len([f for f in os.listdir(work_dir) if f.endswith('.jpg')])
    return (audio_path if os.path.exists(audio_path) else None), n_frames

def select_valid_chunks(frames_dir, n_frames, max_chunks):
    """Duyet tung giay, giu lai toi da max_chunks doan hop le. Tra ve list (sec, box)."""
    valid = []
    for sec in range(n_frames // 30):
        if max_chunks and len(valid) >= max_chunks:
            break
        box, why = analyze_chunk(frames_dir, sec * 30 + 1)
        _stat(why)
        if box is not None:
            valid.append((sec, box))
    return valid


def preprocess_video_simple(video_path, output_base_dir, label, max_chunks=None):
    """
    Moi video: tach frame + audio (ffmpeg, 1 lan) -> bo cac doan 1 giay co CHUYEN CANH hoac KHONG CO MAT
    -> voi doan hop le: crop mat (cung 1 box cho ca 30 frame), resize 224x224 -> luu frame + wav 1 giay.
    Ghi vao thu muc tam roi doi ten (neu bi ngat giua chung se khong de lai video dang do).
    """
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    label_dir = os.path.join(output_base_dir, 'pytmp', label)
    final_dir = os.path.join(label_dir, video_name)
    tmp_dir = os.path.join(label_dir, '.' + video_name + '.tmp')

    if os.path.isdir(final_dir) and len(os.listdir(final_dir)) > 0:
        return

    work = os.path.join(output_base_dir, 'temp_frames', video_name)
    shutil.rmtree(work, ignore_errors=True)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    os.makedirs(tmp_dir, exist_ok=True)

    try:
        audio_path, n_frames = extract_frames_audio(video_path, work, SCAN_SECONDS)
        if audio_path is None:                      # video khong co audio -> bo
            _stat('video_khong_audio')
            return
        with wave.open(audio_path, 'rb') as w:
            params = (w.getnchannels(), w.getsampwidth(), w.getframerate())
            data = w.readframes(w.getnframes())
        bytes_per_sec = params[0] * params[1] * params[2]

        chunk_count = 0
        for sec, box in select_valid_chunks(work, n_frames, max_chunks):
            seg = data[sec * bytes_per_sec:(sec + 1) * bytes_per_sec]
            if len(seg) == 0:
                break
            if len(seg) < bytes_per_sec:             # doan cuoi thieu -> chen im lang cho du 1 giay
                seg = seg + b'\x00' * (bytes_per_sec - len(seg))

            chunk_id = '%05d' % sec
            chunk_dir = os.path.join(tmp_dir, chunk_id)
            os.makedirs(chunk_dir, exist_ok=True)
            crops = crop_chunk(work, sec * 30 + 1, box)
            for k, img in enumerate(crops):
                cv2.imwrite(os.path.join(chunk_dir, '%06d.jpg' % (sec * 30 + 1 + k)), img,
                            [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            with wave.open(os.path.join(tmp_dir, chunk_id + '.wav'), 'wb') as ww:
                ww.setnchannels(params[0]); ww.setsampwidth(params[1]); ww.setframerate(params[2])
                ww.writeframes(seg)
            chunk_count += 1

        if chunk_count > 0:
            os.replace(tmp_dir, final_dir)           # hoan tat -> doi ten (nguyen tu)
        else:
            _stat('video_khong_con_doan_nao')
    except Exception as e:
        print(f'Error processing {video_name}: {e}')
    finally:
        shutil.rmtree(work, ignore_errors=True)
        shutil.rmtree(tmp_dir, ignore_errors=True)

print('Preprocessing functions defined (face crop + scene-cut filter).')


#### Xem trước vùng crop (chạy thử trước khi xử lý cả dataset)

In ra 4 frame (0, 10, 20, 29) của đoạn hợp lệ đầu tiên ở vài video. Nếu miệng bị cắt mất môi/cằm thì tăng `LIP_SCALE` (chế độ `mouth`: `MOUTH_SCALE`); nếu thấy dư nhiều mũi/má thì giảm. Chỉ chạy được khi `DATASET_DIR` có file `.mp4` rời (dataset dạng zip thì tự bỏ qua).

In [ ]:
import matplotlib.pyplot as plt

def preview_crops(n_videos=4, frame_ids=(0, 10, 20, 29)):
    vids_ = sorted(glob.glob(os.path.join(DATASET_DIR, '**', '*.mp4'), recursive=True))[:n_videos]
    if not vids_:
        print('Khong co file .mp4 roi trong DATASET_DIR (dataset zip?) -> bo qua xem truoc.')
        return
    fig, axes = plt.subplots(len(vids_), len(frame_ids), figsize=(3 * len(frame_ids), 3 * len(vids_)), squeeze=False)
    work = os.path.join(cfg.WORK_DIR, 'preview_tmp')
    for r, vp in enumerate(vids_):
        shutil.rmtree(work, ignore_errors=True)
        crops = None
        try:
            _, n_frames = extract_frames_audio(vp, work, SCAN_SECONDS)
            valid = select_valid_chunks(work, n_frames, 1)
            if valid:
                sec, box = valid[0]
                crops = crop_chunk(work, sec * 30 + 1, box)
        finally:
            shutil.rmtree(work, ignore_errors=True)
        for c, k in enumerate(frame_ids):
            ax = axes[r][c]
            ax.axis('off')
            if crops is not None:
                ax.imshow(cv2.cvtColor(crops[k], cv2.COLOR_BGR2RGB))
                if c == 0:
                    ax.set_title(os.path.basename(vp), fontsize=8, loc='left')
            elif c == 0:
                ax.text(0.5, 0.5, 'khong co doan hop le', ha='center', va='center')
    plt.tight_layout()
    plt.show()
    STATS.clear()      # xoa thong ke do cua buoc xem truoc de khong lam lech thong ke that

print('CROP_MODE =', CROP_MODE)
preview_crops()
if LM_STATS:
    print('Landmark moi:', dict(LM_STATS), '(lm_fallback = frame phai dung o theo mep mieng YuNet)')
LM_STATS.clear()


### Chạy tiền xử lý từ Drive

- **Dataset dạng zip:** mở file zip, đọc `metadata.json` (trong zip, hoặc file `*metadata*.json` nằm cạnh zip), rồi chỉ giải nén **từng video cần dùng** ra ổ tạm, xử lý xong thì xóa. Nhờ vậy không bao giờ phải giải nén cả file 9 GB.
- **Dataset dạng thư mục đã giải nén:** đọc video trực tiếp.
- **Chạy lại được:** video đã xử lý trước đó sẽ được tính vào số lượng và bỏ qua, nên nếu Colab ngắt giữa chừng chỉ cần chạy lại cell này.

In [ ]:
import zipfile

TMP_DIR = '/content/tmp_videos'
train_dir = cfg.TRAIN_DIR
os.makedirs(train_dir, exist_ok=True)

def _done(label):
    d = os.path.join(train_dir, 'pytmp', label)
    if not os.path.isdir(d):
        return set()
    for name in os.listdir(d):                 # don dep thu muc tam do lan chay truoc bi ngat
        if name.startswith('.'):
            shutil.rmtree(os.path.join(d, name), ignore_errors=True)
    return {n for n in os.listdir(d) if os.listdir(os.path.join(d, n))}

done = {'real': _done('real'), 'fake': _done('fake')}
caps = {'real': MAX_REAL, 'fake': MAX_FAKE}
counts = {k: len(v) for k, v in done.items()}
print('Da co san tu truoc:', counts)

def need(label):
    return caps[label] is None or counts[label] < caps[label]

def all_full():
    return (not need('real')) and (not need('fake'))

def process(label, stem, get_path, cleanup=None):
    if stem in done[label] or not need(label):
        return
    path = get_path()
    if path is None or not os.path.exists(path):
        return
    preprocess_video_simple(path, train_dir, label, max_chunks=MAX_CHUNKS)
    if cleanup:
        cleanup(path)
    pdir = os.path.join(train_dir, 'pytmp', label, stem)
    if os.path.isdir(pdir) and os.listdir(pdir):
        counts[label] += 1
        done[label].add(stem)

def label_of(info):
    return 'real' if info['label'] == 'REAL' else 'fake'

# ---------- Che do 1: dataset la cac file zip ----------
for zpath in zips:
    if all_full():
        break
    print(f'\n== {os.path.basename(zpath)} ==  real={counts["real"]} fake={counts["fake"]}')
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    os.makedirs(TMP_DIR, exist_ok=True)
    with zipfile.ZipFile(zpath) as zf:
        names = set(zf.namelist())
        meta_members = [n for n in names if n.endswith('metadata.json')]
        entries = []   # (member_trong_zip, label)
        if meta_members:
            for mm in meta_members:
                meta = json.loads(zf.read(mm).decode('utf-8'))
                base = os.path.dirname(mm)
                for vname, info in meta.items():
                    member = f'{base}/{vname}' if base else vname
                    if member in names:
                        entries.append((member, label_of(info)))
        else:
            ext = sorted(glob.glob(os.path.join(os.path.dirname(zpath), '*metadata*.json')) +
                         glob.glob(os.path.join(DATASET_DIR, '*metadata*.json')))
            if not ext:
                print('  Khong co metadata.json trong zip va khong thay file metadata ben ngoai -> bo qua zip nay')
                continue
            meta = json.load(open(ext[0]))
            by_base = {os.path.basename(n): n for n in names if n.lower().endswith('.mp4')}
            for vname, info in meta.items():
                if vname in by_base:
                    entries.append((by_base[vname], label_of(info)))
        print(f'  {len(entries)} video co nhan trong zip')

        for member, label in tqdm(entries, desc='  xu ly'):
            if all_full():
                break
            stem = os.path.splitext(os.path.basename(member))[0]
            process(label, stem,
                    get_path=lambda m=member: zf.extract(m, TMP_DIR),
                    cleanup=os.remove)
    shutil.rmtree(TMP_DIR, ignore_errors=True)

# ---------- Che do 2: khong co zip -> thu muc chua video + file json (chay song song) ----------
if not zips:
    _index = {}
    def find_video(base, vname):
        p = os.path.join(base, vname)
        if os.path.exists(p):
            return p
        if not _index:   # video nam o thu muc con khac -> lap chi muc theo ten file
            for v in glob.glob(os.path.join(DATASET_DIR, '**', '*.mp4'), recursive=True):
                _index[os.path.basename(v)] = v
        return _index.get(vname)

    from concurrent.futures import ThreadPoolExecutor, as_completed
    for mpath in metas:
        if all_full():
            break
        try:
            meta = json.load(open(mpath))
        except Exception:
            continue
        sample = list(meta.values())[:20] if isinstance(meta, dict) else []
        if not sample or not all(isinstance(v, dict) and 'label' in v for v in sample):
            print('Bo qua (khong phai file metadata co nhan):', mpath)
            continue
        base = os.path.dirname(mpath)

        # 1) Chon truoc danh sach video can xu ly (ap dung gioi han MAX_REAL / MAX_FAKE)
        scheduled = dict(counts)
        tasks = []
        for vname, info in meta.items():
            label = label_of(info)
            stem = os.path.splitext(vname)[0]
            if stem in done[label]:
                continue
            if caps[label] is not None and scheduled[label] >= caps[label]:
                continue
            path = find_video(base, vname)
            if path is None:
                continue
            tasks.append((label, stem, path))
            scheduled[label] += 1
        print(f'\n== {mpath} ==  {len(meta)} video trong json, can xu ly {len(tasks)}, song song {WORKERS} luong')

        # 2) Chay song song
        ex = ThreadPoolExecutor(max_workers=WORKERS)
        try:
            futs = {ex.submit(preprocess_video_simple, p, train_dir, l, MAX_CHUNKS): (l, st) for l, st, p in tasks}
            for f in tqdm(as_completed(futs), total=len(futs), desc='  xu ly'):
                l, st = futs[f]
                try:
                    f.result()
                except Exception as e:
                    print('Loi:', st, e)
                pdir = os.path.join(train_dir, 'pytmp', l, st)
                if os.path.isdir(pdir) and os.listdir(pdir):
                    counts[l] += 1
                    done[l].add(st)
        except KeyboardInterrupt:
            print('Da dung. Chay lai cell de tiep tuc.')
        finally:
            ex.shutdown(wait=False, cancel_futures=True)

print(f'\nTong da xu ly: {counts["real"]} real, {counts["fake"]} fake video')
for temp_dir in ['temp_frames', 'temp_audio']:
    temp_path = os.path.join(cfg.TRAIN_DIR, temp_dir)
    if os.path.exists(temp_path):
        shutil.rmtree(temp_path)

In [ ]:
# Thong ke ly do loai doan (chay sau khi tien xu ly xong)
tot = sum(STATS.values())
print('Tong so lan xet doan 1 giay:', tot)
for k, v in STATS.most_common():
    print(f'  {k:28s} {v:7d}  ({v/max(tot,1)*100:5.1f}%)')
# Neu 'chuyen_canh' hoac 'khong_co_mat' qua cao: tang CUT_THRESH / giam MIN_FACE_FRAC / tang SCAN_SECONDS o cell tren roi chay lai.
if LM_STATS:
    _n = sum(LM_STATS.values())
    print(f'\nLandmark moi (MediaPipe): {dict(LM_STATS)}  -> fallback {LM_STATS["lm_fallback"] / max(_n, 1) * 100:.1f}% frame mau')
    # fallback cao (>20%) nghia la MediaPipe hay khong bat duoc moi (mat nho/nghieng/che): xem lai bang cell xem truoc

## 5. Chia train / valid / test (theo video) và lưu CSV

So với bản gốc: danh sách video được **sắp xếp (`sorted`) trước khi shuffle** để kết quả chia luôn giống nhau với cùng `SEED`, và file CSV sẽ được lưu cố định vào file `.tar`, nên notebook train không bao giờ chia lại.

In [ ]:
def write_csv_list(data_list, path):
    with open(path, 'w', newline='') as f:
        writer = csv.writer(f, delimiter=',')
        for row in data_list:
            if row:
                writer.writerow(row)
    print(f'Split saved to {path} ({len(data_list)} entries)')


def generate_splits(work_dir, train_ratio=0.7, valid_ratio=0.15):
    """
    Split preprocessed videos into train / valid / test at the VIDEO level
    (so chunks from the same source video never leak across splits), then
    write out the chunk-level rows for each split to CSV.

    train_ratio + valid_ratio must leave room for a test split
    (test_ratio = 1 - train_ratio - valid_ratio).
    """
    assert 0 < train_ratio < 1, 'train_ratio must be between 0 and 1'
    assert 0 < valid_ratio < 1, 'valid_ratio must be between 0 and 1'
    assert train_ratio + valid_ratio < 1, 'train_ratio + valid_ratio must leave room for a test split'

    pytmp_real = os.path.join(work_dir, 'train', 'pytmp', 'real')
    pytmp_fake = os.path.join(work_dir, 'train', 'pytmp', 'fake')

    real_videos = []
    fake_videos = []

    if os.path.exists(pytmp_real):
        real_videos = [d for d in sorted(os.listdir(pytmp_real))
                       if os.path.isdir(os.path.join(pytmp_real, d))]
    if os.path.exists(pytmp_fake):
        fake_videos = [d for d in sorted(os.listdir(pytmp_fake))
                       if os.path.isdir(os.path.join(pytmp_fake, d))]

    print(f'Found {len(real_videos)} real videos, {len(fake_videos)} fake videos')

    random.shuffle(real_videos)
    random.shuffle(fake_videos)

    def three_way_split(videos, train_ratio, valid_ratio):
        n = len(videos)
        n_train = int(n * train_ratio)
        n_valid = int(n * valid_ratio)
        train = videos[:n_train]
        valid = videos[n_train:n_train + n_valid]
        test = videos[n_train + n_valid:]
        return train, valid, test

    real_train, real_valid, real_test = three_way_split(real_videos, train_ratio, valid_ratio)
    fake_train, fake_valid, fake_test = three_way_split(fake_videos, train_ratio, valid_ratio)

    print(f'Train: {len(real_train)} real, {len(fake_train)} fake')
    print(f'Valid: {len(real_valid)} real, {len(fake_valid)} fake')
    print(f'Test : {len(real_test)} real, {len(fake_test)} fake')

    base_train = os.path.join('train', 'pytmp')

    def collect_rows(video_names, pytmp_dir, label):
        rows = []
        for video_name in video_names:
            video_dir = os.path.join(pytmp_dir, video_name)
            if not os.path.isdir(video_dir):
                continue
            for chunk in sorted(os.listdir(video_dir)):
                chunk_path = os.path.join(video_dir, chunk)
                if os.path.isdir(chunk_path):
                    audio_file = chunk + '.wav'
                    audio_path = os.path.join(video_dir, audio_file)
                    if os.path.exists(audio_path):
                        rows.append([
                            os.path.join(base_train, label, video_name, chunk),
                            os.path.join(base_train, label, video_name, audio_file),
                            label
                        ])
        return rows

    train_set = collect_rows(fake_train, pytmp_fake, 'fake') + collect_rows(real_train, pytmp_real, 'real')
    valid_set = collect_rows(fake_valid, pytmp_fake, 'fake') + collect_rows(real_valid, pytmp_real, 'real')
    test_set = collect_rows(fake_test, pytmp_fake, 'fake') + collect_rows(real_test, pytmp_real, 'real')

    random.shuffle(train_set)
    random.shuffle(valid_set)
    random.shuffle(test_set)

    write_csv_list(train_set, os.path.join(work_dir, 'train_split.csv'))
    write_csv_list(valid_set, os.path.join(work_dir, 'valid_split.csv'))
    write_csv_list(test_set, os.path.join(work_dir, 'test_imbalance_split.csv'))

    return len(train_set), len(valid_set), len(test_set)

random.seed(cfg.SEED)   # co dinh seed de tap valid/test khong doi moi lan chay lai
n_train, n_valid, n_test = generate_splits(cfg.WORK_DIR, train_ratio=cfg.TRAIN_RATIO, valid_ratio=cfg.VALID_RATIO)
print(f'\nTrain samples: {n_train}, Valid samples: {n_valid}, Test samples: {n_test}')

In [ ]:
# ====================== KIEM TRA CAN BANG LOP SAU KHI CHIA SPLIT ======================
import pandas as pd

def check_split_balance(csv_path, split_name):
    df = pd.read_csv(csv_path, header=None, names=['video_path', 'audio_path', 'label'])
    counts = df['label'].value_counts()
    n_real = counts.get('real', 0)
    n_fake = counts.get('fake', 0)
    total = n_real + n_fake
    n_unique_videos = df['video_path'].apply(lambda p: p.split('/')[-2]).nunique()
    print(f'{split_name:>6} | Real: {n_real:5d} ({n_real/total*100:5.1f}%)  '
          f'Fake: {n_fake:5d} ({n_fake/total*100:5.1f}%)  '
          f'Tong chunk: {total:5d}  |  So video goc: {n_unique_videos}')

print('Ti le real/fake tren tung tap sau khi chia (theo chunk, khong phai theo video):')
print('-' * 90)
check_split_balance(os.path.join(cfg.WORK_DIR, 'train_split.csv'), 'Train')
check_split_balance(os.path.join(cfg.WORK_DIR, 'valid_split.csv'), 'Valid')
check_split_balance(os.path.join(cfg.WORK_DIR, 'test_imbalance_split.csv'), 'Test')

## 6. Nén và lưu lên Drive

Nén ở ổ cục bộ trước (nhanh, ít lỗi), sau đó mới copy **một file lớn** lên Drive. Không copy từng file nhỏ lên Drive vì rất chậm.

In [ ]:
TAR_LOCAL = '/content/dfdc_processed.tar'
TAR_DRIVE = os.path.join(cfg.DRIVE_DIR, 'dfdc_processed.tar')

# 1) Dam bao du file split (bo qua neu da dong tar xong o lan chay truoc)
if not os.path.exists(TAR_LOCAL):
    for f in ['train_split.csv', 'valid_split.csv', 'test_imbalance_split.csv']:
        assert os.path.exists(os.path.join(cfg.WORK_DIR, f)), f'Thieu {f}'

    # 2) Nen o o cuc bo. --remove-files: xoa tung file goc ngay sau khi da vao tar,
    #    nen dinh dung luong chi con ~1 ban du lieu (thay vi pytmp + tar = 2 ban).
    #    Luu y: sau buoc nay pytmp va 3 file CSV trong WORK_DIR bi xoa, ban duy nhat la file tar.
    !tar -cf {TAR_LOCAL} --remove-files -C {cfg.WORK_DIR} train/pytmp train_split.csv valid_split.csv test_imbalance_split.csv
else:
    print('Da co file tar local tu lan chay truoc -> bo qua buoc nen.')

size_local = os.path.getsize(TAR_LOCAL)
print(f'Tar cuc bo: {size_local/1024**3:.2f} GB')

# 3) Kiem tra Drive con du cho khong (neu thieu: giai phong Drive roi chay lai tu cell sau; tar local van con)
free = shutil.disk_usage('/content/drive/MyDrive').free
print(f'Drive con trong: {free/1024**3:.2f} GB')
assert free > size_local * 1.05, 'Drive khong du dung luong! Hay giai phong Drive roi chay tiep cell copy (file tar local van con nguyen).'


In [ ]:
import shutil
import os

# Tạo thư mục đích trên Drive nếu chưa có
os.makedirs(os.path.dirname(TAR_DRIVE), exist_ok=True)

# Tiến hành copy file từ cục bộ lên Google Drive
print('Đang copy file tar lên Google Drive (quá trình này có thể mất vài phút)...')
shutil.copy(TAR_LOCAL, TAR_DRIVE)

# Kiểm tra kích thước
size_local = os.path.getsize(TAR_LOCAL)
size_drive = os.path.getsize(TAR_DRIVE)
print(f'Kích thước cục bộ: {size_local/1024**3:.2f} GB')
print(f'Kích thước trên Drive: {size_drive/1024**3:.2f} GB')

assert size_drive == size_local, 'Lỗi: Kích thước file không khớp sau khi copy! (file tar local được giữ lại, hãy chạy lại cell này)'

# Đọc thử một vài dòng đầu tiên từ file tar trên Drive
print('\nDanh sách một số file đầu tiên trong file nén trên Drive:')
!tar -tf {TAR_DRIVE} | head -5
print('\nĐã lưu xong hoàn toàn lên Drive tại:', TAR_DRIVE)

# Drive da co ban day du va da kiem tra -> xoa tar local de giai phong o dia
os.remove(TAR_LOCAL)
print('Da xoa file tar local, giai phong o dia.')


In [ ]:
# Ghi het du lieu len Drive truoc khi roi notebook (nen chay cell nay cuoi cung)
from google.colab import drive
drive.flush_and_unmount()
print('Drive da flush. Co the chuyen sang notebook 2.')